# ogbl-collab Training, Validation, and Test Visualizations

This notebook visualizes the training logs produced by `scripts/train.py`. It focuses on loss, learning rate, validation/test Hits@K, best-checkpoint behavior, and the separate results sheet.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

ROOT = Path('..').resolve()
LOG_CANDIDATES = [
    ROOT / 'logs/default/training_log.json',
    ROOT / 'logs/training_log.json',
    ROOT / 'logs/structural/training_log.json',
    ROOT / 'logs/quick/training_log.json',
    ROOT / 'logs/smoke/training_log.json',
]
CONFIG_CANDIDATES = [
    ROOT / 'logs/default/config_used.yaml',
    ROOT / 'logs/config_used.yaml',
    ROOT / 'configs/default.yaml',
]
RESULTS_PATH = ROOT / 'results/results.csv'

LOG_PATH = next((path for path in LOG_CANDIDATES if path.exists()), None)
CONFIG_PATH = next((path for path in CONFIG_CANDIDATES if path.exists()), None)

if LOG_PATH is None:
    raise FileNotFoundError('No training log found. Run `uv run python scripts/train.py` first.')

log = json.loads(LOG_PATH.read_text())
config = yaml.safe_load(CONFIG_PATH.read_text()) if CONFIG_PATH else {}
results_sheet = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame()

print(f'Loaded log: {LOG_PATH.relative_to(ROOT)}')
if CONFIG_PATH:
    print(f'Loaded config: {CONFIG_PATH.relative_to(ROOT)}')
print(f'Runs in log: {len(log.get("records", []))}')

## Parse Logs

In [ ]:
def build_loss_frame(log):
    rows = []
    for run_idx, losses in enumerate(log.get('losses', []), start=1):
        for epoch, loss in enumerate(losses, start=1):
            rows.append({'run': run_idx, 'epoch': epoch, 'loss': loss})
    return pd.DataFrame(rows)


def build_eval_frame(log):
    rows = []
    for run_idx, records in enumerate(log.get('records', []), start=1):
        for record in records:
            epoch = record.get('epoch')
            lr = record.get('lr')
            loss = record.get('loss')
            for metric, splits in record.get('metrics', {}).items():
                for split, value in splits.items():
                    rows.append({
                        'run': run_idx,
                        'epoch': epoch,
                        'metric': metric,
                        'split': split,
                        'value': value,
                        'value_percent': 100 * value,
                        'lr': lr,
                        'loss': loss,
                    })
    return pd.DataFrame(rows)


def build_best_frame(log):
    rows = []
    for metric, best_by_run in log.get('best', {}).items():
        for run_idx, item in enumerate(best_by_run, start=1):
            if item.get('epoch_index', -1) < 0:
                continue
            rows.append({
                'metric': metric,
                'run': run_idx,
                'best_valid_percent': 100 * item.get('best_valid', np.nan),
                'test_at_best_valid_percent': 100 * item.get('test_at_best_valid', np.nan),
                'eval_index': item.get('epoch_index'),
            })
    return pd.DataFrame(rows)


loss_df = build_loss_frame(log)
eval_df = build_eval_frame(log)
best_df = build_best_frame(log)

display(loss_df.head())
display(eval_df.head())
display(best_df)

## Experiment Overview

In [ ]:
overview_rows = []
for section in ['dataset', 'model', 'predictor', 'structural_features', 'training', 'evaluation', 'early_stopping', 'experiment']:
    values = config.get(section, {}) if isinstance(config, dict) else {}
    if isinstance(values, dict):
        for key, value in values.items():
            overview_rows.append({'section': section, 'key': key, 'value': value})

overview_df = pd.DataFrame(overview_rows)
display(overview_df)

summary = {
    'runs': len(log.get('records', [])),
    'logged_loss_points': len(loss_df),
    'logged_eval_points': len(eval_df),
    'metrics': ', '.join(sorted(eval_df['metric'].unique())) if not eval_df.empty else '',
    'splits': ', '.join(sorted(eval_df['split'].unique())) if not eval_df.empty else '',
}
display(pd.DataFrame([summary]))

## Results Sheet

In [ ]:
if results_sheet.empty:
    print('No completed result rows yet. The sheet exists after setup and is appended after training finishes.')
else:
    display(results_sheet)
    numeric_cols = ['Test Hits@50', 'Validation Hits@50']
    plot_df = results_sheet.copy()
    for col in numeric_cols:
        plot_df[col] = pd.to_numeric(plot_df[col].astype(str).str.split().str[0], errors='coerce')
    long = plot_df.melt(id_vars=['Method', 'Date'], value_vars=numeric_cols, var_name='score', value_name='Hits@50')
    plt.figure(figsize=(10, 4))
    sns.barplot(data=long, x='Method', y='Hits@50', hue='score')
    plt.xticks(rotation=20, ha='right')
    plt.title('Recorded Results Sheet')
    plt.tight_layout()
    plt.show()

## Training Loss

In [ ]:
if loss_df.empty:
    print('No training loss values found.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    sns.lineplot(data=loss_df, x='epoch', y='loss', hue='run', marker='o', ax=axes[0], palette='tab10')
    axes[0].set_title('Training Loss by Run')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')

    smooth_df = loss_df.copy()
    smooth_df['ema_loss'] = smooth_df.groupby('run')['loss'].transform(lambda s: s.ewm(alpha=0.15, adjust=False).mean())
    sns.lineplot(data=smooth_df, x='epoch', y='ema_loss', hue='run', ax=axes[1], palette='tab10')
    axes[1].set_title('Smoothed Training Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('EMA Loss')
    plt.tight_layout()
    plt.show()

## Learning Rate Schedule

In [ ]:
lr_df = eval_df[['run', 'epoch', 'lr']].drop_duplicates().dropna() if not eval_df.empty else pd.DataFrame()
if lr_df.empty:
    print('No learning-rate records found.')
else:
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=lr_df, x='epoch', y='lr', hue='run', marker='o', palette='tab10')
    plt.yscale('log')
    plt.title('Learning Rate at Evaluation Steps')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.tight_layout()
    plt.show()

## Hits@K Curves by Split

In [ ]:
if eval_df.empty:
    print('No evaluation metrics found.')
else:
    metrics = sorted(eval_df['metric'].unique(), key=lambda x: int(x.split('@')[1]))
    fig, axes = plt.subplots(len(metrics), 1, figsize=(12, 4 * len(metrics)), sharex=True)
    if len(metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, metrics):
        frame = eval_df[eval_df['metric'] == metric]
        sns.lineplot(data=frame, x='epoch', y='value_percent', hue='split', style='run', markers=True, ax=ax)
        ax.set_title(metric)
        ax.set_ylabel('Hits (%)')
        ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.show()

## Mean and Variability Across Runs

In [ ]:
if eval_df.empty:
    print('No evaluation metrics found.')
else:
    agg = eval_df.groupby(['epoch', 'metric', 'split'], as_index=False).agg(
        mean_percent=('value_percent', 'mean'),
        std_percent=('value_percent', 'std'),
    )
    agg['std_percent'] = agg['std_percent'].fillna(0.0)
    for metric in sorted(agg['metric'].unique(), key=lambda x: int(x.split('@')[1])):
        fig, ax = plt.subplots(figsize=(11, 4))
        subset = agg[agg['metric'] == metric]
        for split, frame in subset.groupby('split'):
            ax.plot(frame['epoch'], frame['mean_percent'], marker='o', label=split)
            ax.fill_between(frame['epoch'], frame['mean_percent'] - frame['std_percent'], frame['mean_percent'] + frame['std_percent'], alpha=0.15)
        ax.set_title(f'{metric}: Mean +/- Std Across Runs')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Hits (%)')
        ax.legend()
        plt.tight_layout()
        plt.show()

## Best Checkpoint Summary

In [ ]:
if best_df.empty:
    print('No best-checkpoint data found.')
else:
    display(best_df.sort_values(['metric', 'run']))
    plot_df = best_df.melt(
        id_vars=['metric', 'run', 'eval_index'],
        value_vars=['best_valid_percent', 'test_at_best_valid_percent'],
        var_name='score_type',
        value_name='Hits (%)',
    )
    plt.figure(figsize=(12, 5))
    sns.barplot(data=plot_df, x='metric', y='Hits (%)', hue='score_type')
    plt.title('Best Validation and Test-at-Best-Validation')
    plt.tight_layout()
    plt.show()

## Validation/Test Alignment

In [ ]:
if eval_df.empty:
    print('No evaluation metrics found.')
else:
    pivot = eval_df.pivot_table(index=['run', 'epoch', 'metric'], columns='split', values='value_percent').reset_index()
    if {'valid', 'test'}.issubset(pivot.columns):
        g = sns.relplot(data=pivot, x='valid', y='test', hue='metric', style='run', col='metric', col_wrap=3, height=4, facet_kws={'sharex': False, 'sharey': False})
        g.fig.suptitle('Validation vs Test Hits at Evaluation Steps', y=1.03)
        for ax in g.axes.flat:
            lo = min(ax.get_xlim()[0], ax.get_ylim()[0])
            hi = max(ax.get_xlim()[1], ax.get_ylim()[1])
            ax.plot([lo, hi], [lo, hi], color='gray', linestyle='--', linewidth=1)
        plt.show()
    else:
        print('Validation/test splits are not both present.')

## Generalization Gap

In [ ]:
if eval_df.empty:
    print('No evaluation metrics found.')
else:
    pivot = eval_df.pivot_table(index=['run', 'epoch', 'metric'], columns='split', values='value_percent').reset_index()
    if {'train', 'valid', 'test'}.issubset(pivot.columns):
        pivot['train_minus_valid'] = pivot['train'] - pivot['valid']
        pivot['valid_minus_test'] = pivot['valid'] - pivot['test']
        gap_df = pivot.melt(
            id_vars=['run', 'epoch', 'metric'],
            value_vars=['train_minus_valid', 'valid_minus_test'],
            var_name='gap',
            value_name='percentage_points',
        )
        g = sns.relplot(data=gap_df, x='epoch', y='percentage_points', hue='gap', style='run', col='metric', col_wrap=3, kind='line', marker='o', height=4)
        g.fig.suptitle('Generalization Gap Over Time', y=1.03)
        for ax in g.axes.flat:
            ax.axhline(0, color='gray', linewidth=1, linestyle='--')
        plt.show()
    else:
        print('Train/valid/test splits are not all present.')

## Final Evaluation Heatmap

In [ ]:
if eval_df.empty:
    print('No evaluation metrics found.')
else:
    last_epochs = eval_df.groupby('run')['epoch'].max().rename('last_epoch').reset_index()
    final_df = eval_df.merge(last_epochs, on='run')
    final_df = final_df[final_df['epoch'] == final_df['last_epoch']]
    heat = final_df.pivot_table(index='metric', columns='split', values='value_percent', aggfunc='mean')
    plt.figure(figsize=(7, 4))
    sns.heatmap(heat, annot=True, fmt='.2f', cmap='viridis')
    plt.title('Final Logged Evaluation Mean Hits (%)')
    plt.tight_layout()
    plt.show()

## Checkpoint Inventory

In [ ]:
checkpoint_dirs = [ROOT / 'checkpoints/default', ROOT / 'checkpoints']
rows = []
for directory in checkpoint_dirs:
    if not directory.exists():
        continue
    for path in sorted(directory.glob('**/*.pt')):
        rows.append({
            'path': str(path.relative_to(ROOT)),
            'size_mb': path.stat().st_size / (1024 ** 2),
            'modified': pd.to_datetime(path.stat().st_mtime, unit='s'),
        })

checkpoint_df = pd.DataFrame(rows).drop_duplicates('path') if rows else pd.DataFrame()
if checkpoint_df.empty:
    print('No checkpoints found.')
else:
    display(checkpoint_df)
    plt.figure(figsize=(10, 3))
    sns.barplot(data=checkpoint_df, x='path', y='size_mb')
    plt.xticks(rotation=30, ha='right')
    plt.title('Checkpoint Sizes')
    plt.ylabel('MB')
    plt.tight_layout()
    plt.show()